In [ ]:
import os, json, time
from pathlib import Path
from urllib.request import urlopen
from urllib.error import URLError, HTTPError
import numpy as np
import pandas as pd

In [ ]:
SEOUL_API_KEY = "-"

PROJECT_ROOT = Path('.') 
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
EXT_DIR = PROJECT_ROOT / 'data' / 'external'
RAW_DIR.mkdir(parents=True, exist_ok=True)
EXT_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = 'http://openapi.seoul.go.kr:8088'
QUARTERS = [f'{y}{q}' for y in range(2020, 2026) for q in range(1, 5)]
QUARTERS.append('20261')

In [ ]:
def fetch_one_page(endpoint, start, end, extra=''):
    suffix = f'/{extra}' if extra else ''
    url = f'{BASE_URL}/{SEOUL_API_KEY}/json/{endpoint}/{start}/{end}{suffix}'
    for attempt in range(3):
        try:
            with urlopen(url, timeout=30) as resp:
                payload = json.loads(resp.read().decode('utf-8'))
            block = payload.get(endpoint, {})
            result = block.get('RESULT', {})
            if result and result.get('CODE') not in ('INFO-000', None):
                return []
            return block.get('row', [])
        except (URLError, HTTPError, json.JSONDecodeError) as e:
            if attempt == 2:
                print(f'호출 실패 ({endpoint}, {extra}): {e}')
                return []
            time.sleep(1.5)
    return []

def fetch_all(endpoint, extra='', max_rows=50000):
    rows = []
    for start in range(1, max_rows + 1, 1000):
        end = start + 999
        page = fetch_one_page(endpoint, start, end, extra)
        if not page:
            break
        rows.extend(page)
        if len(page) < 1000:
            break
    return rows

In [ ]:
MAPPING_PATH = RAW_DIR / 'TbgisTrdarRelm.csv'

if MAPPING_PATH.exists():
    df_map = pd.read_csv(MAPPING_PATH, dtype=str)
    print(f'캐시 사용: {len(df_map)}건')
else:
    rows = fetch_all('TbgisTrdarRelm', max_rows=2000)
    df_map = pd.DataFrame(rows)
    df_map.to_csv(MAPPING_PATH, index=False, encoding='utf-8-sig')
    print(f'수집 완료: {len(df_map)}건')

# 자치구 25개 확인
print(sorted(df_map['SIGNGU_CD_NM'].unique()))

In [ ]:
# 딱 3분기만 받아서 분기 필터가 작동하는지 확인
test_quarters = ['20231', '20232', '20233']
for q in test_quarters:
    rows = fetch_all('VwsmAdstrdSelngQq', extra=q, max_rows=1000)
    df_test = pd.DataFrame(rows)
    if not df_test.empty and 'STDR_YYQU_CD' in df_test.columns:
        print(f'{q} → 실제 분기값: {df_test["STDR_YYQU_CD"].unique()}')
    else:
        print(f'{q} → 빈 응답')